<a href="https://colab.research.google.com/github/velchan15/MachineLearning-InternshipStarter-FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **Refresh / Content Opportunity Scoring**.

The core question is
"which pages should be reviewed first?" — a "which one first" question, which maps to a **scoring / ranking** task. Under the hood there is a binary classifier predicting the probability that a page is declining, but the final output used by an editor is a ranked list, not a single yes/no answer.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is `is_declining_label`: 1 when `trend_direction == "down"`, i.e. trailing-30-day impressions dropped more than 20% versus the prior 30 days. This is an **observed** outcome (a real measured change in traffic), not a label defined by someone's opinion — so it satisfies the rule that a target must be observed, not defined. `trend_direction` and `trend_pct` are the source of this label and must never be used as model features, or the model would just be learning the label-generating rule (leakage).

In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/velchan15/MachineLearning-InternshipStarter-FlyRank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["is_declining_label"].value_counts(normalize=True)

,proportion
is_declining_label,
1,0.542067
0,0.457933


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The main metric is **Precision@50**: out of the top 50 pages the model ranks as highest priority, what fraction are genuinely declining. This matches the reference pipeline in this repo, where a hand-written rule scores ~24% and a trained model scores ~74% on this same metric. I will also track **Recall**, since for this use case missing a genuinely declining page (false negative) is more costly than flagging a healthy page for review (false positive).

In [8]:
def precision_at_k(y_true, y_score, k=50):
    top_k_idx = y_score.sort_values(ascending=False).index[:k]
    return y_true.loc[top_k_idx].mean()

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page. The starter dataset has 30,000 rows across 32
pseudonymized clients, with metrics aggregated over a trailing 90-day window.

In [9]:
print(df.shape)
df[["content_id", "client_id", "content_type", "impressions_90d",
    "trend_direction", "is_declining_label"]].head()

(30000, 45)


,content_id,client_id,content_type,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A hand-written rule (e.g. "flag it if the page is older than a year and
traffic dropped") is easy to write but only catches part of the pattern.
Signals like search position, CTR, content type, keyword competition, and
engagement rate interact with each other and shift over time — too tangled for a single if-statement. The reference pipeline in this repo proves this directly: the manual rule reaches ~24% precision on the top 50 pages, while a model trained on the combined signals reaches ~74% — almost 3x better. That gap is exactly what "a real but too-messy-to-hand-write" pattern looks like.

In [10]:
cols = ["avg_position", "ctr", "engagement_rate", "word_count", "content_age_days"]
df[cols + ["is_declining_label"]].corr()["is_declining_label"]

,is_declining_label
avg_position,-0.029035
ctr,-0.061911
engagement_rate,-0.012743
word_count,0.090157
content_age_days,-0.163882
is_declining_label,1.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.